# Finetuning with Azure OpenAI

This notebook demonstrates how to fine-tune language models using **Supervised Fine-Tuning (SFT)**, **Direct Preference Optimization (DPO)**, and **Reinforcement Fine-Tuning (RFT)**.

**Note**: Execute each cell in sequence.

## 1. Setup and Installation

In [ ]:
%pip install -q \
  "azure-ai-projects>=2.0.0b1" \
  openai \
  azure-identity \
  azure-mgmt-cognitiveservices \
  "azure-ai-evaluation>=1.13.0" \
  python-dotenv

## 2. Configure Azure

This sets up the Azure credential authentication and initializes OpenAI client needed for fine-tuning workflows.

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

In [ ]:
# TODO: Update RESOURCE_GROUP name and OPENAI_API_KEY that have been provided to your team via gradescope.
# Do not modify other fields.
RESOURCE_GROUP = 'cis-5270-team-9'
OPENAI_API_KEY = ''
OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ''

In [ ]:
# Azure resource targeting
os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"] = RESOURCE_GROUP
os.environ["AZURE_AOAI_ACCOUNT"] = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT
CREDENTIAL = DefaultAzureCredential()

We're using **gpt-4.1-nano** in this example, but you can use other supported GPT models.

In [ ]:
openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)
model_name = 'gpt-4.1-nano-2025-04-14'
print("Connected to Azure OpenAI")

## 3. Upload Training Files

Upload the training and validation JSONL files to Microsoft Foundry. Each file is assigned a unique ID that will be referenced when creating the fine-tuning job. See [here]() for how dataset should be prepared.

In [ ]:
import json
import re

START_MARKER = "-- FORMAT-START-9F3A"
SIGNATURE_MARKER = "-- FORMAT-SIGNATURE-9F3A"
END_MARKER = "-- FORMAT-END-9F3A"

INPUT_FILES = {
    "training.jsonl": "training_sft_sig_marker.jsonl",
    "validation.jsonl": "validation_sft_sig_marker.jsonl",
    "test.jsonl": "test_sft_sig_marker.jsonl",
}

def add_constraint(prompt):
    return (
        prompt.strip()
        + "\n\nFormatting constraints:\n"
        + "1. The first non-empty line of your solution must be exactly:\n"
        + f"{START_MARKER}\n"
        + "2. Immediately after the first top-level type signature in your solution, include this exact line:\n"
        + f"{SIGNATURE_MARKER}\n"
        + "3. The last non-empty line of your solution must be exactly:\n"
        + f"{END_MARKER}"
    )

def is_top_level_signature(line):
    s = line.strip()
    if s.startswith("--") or s.startswith("{-") or s.startswith("import ") or s.startswith("module "):
        return False
    return re.match(r"^[A-Za-z_][A-Za-z0-9_']*\s*::", s) is not None

def add_markers(solution):
    lines = solution.strip().splitlines()
    out = [START_MARKER]
    inserted = False

    for line in lines:
        out.append(line)
        if not inserted and is_top_level_signature(line):
            out.append(SIGNATURE_MARKER)
            inserted = True

    if not inserted:
        # fallback: still include the marker so the target has the required format
        out.append(SIGNATURE_MARKER)

    out.append(END_MARKER)
    return "\n".join(out)

for in_path, out_path in INPUT_FILES.items():
    written = 0
    skipped = 0

    with open(in_path, "r", encoding="utf-8") as f_in, \
         open(out_path, "w", encoding="utf-8") as f_out:

        for line in f_in:
            ex = json.loads(line)

            if "messages" in ex and "reference_solution" not in ex:
                msgs = ex["messages"]

                if len(msgs) < 2:
                    skipped += 1
                    continue

                user_msg = msgs[0]["content"]
                assistant_msg = msgs[1]["content"]

                new_ex = {
                    "messages": [
                        {"role": "user", "content": add_constraint(user_msg)},
                        {"role": "assistant", "content": add_markers(assistant_msg)},
                    ]
                }

            elif "prompt" in ex and "reference_solution" in ex:
                new_ex = dict(ex)
                new_ex["prompt"] = add_constraint(ex["prompt"])
                new_ex["reference_solution"] = add_markers(ex["reference_solution"])

            else:
                skipped += 1
                continue

            f_out.write(json.dumps(new_ex, ensure_ascii=False) + "\n")
            written += 1

    print(f"Wrote {out_path}: {written} examples, skipped {skipped}")

In [ ]:
training_file_path = "training_sft_sig_marker.jsonl"
validation_file_path = "validation_sft_sig_marker.jsonl"

print("Uploading training file (signature marker SFT)...")
with open(training_file_path, "rb") as f:
    train_file = openai_client.files.create(
        file=f,
        purpose="fine-tune"
    )

print("Uploading validation file (signature marker SFT)...")
with open(validation_file_path, "rb") as f:
    validation_file = openai_client.files.create(
        file=f,
        purpose="fine-tune"
    )

train_file_id = train_file.id
val_file_id = validation_file.id

print(f"Training file ID: {train_file_id}")
print(f"Validation file ID: {val_file_id}")

Microsoft Foundry needs to process the uploaded files before they can be used for fine-tuning.

In [ ]:
print("Waiting for files to be processed...")
openai_client.files.wait_for_processing(train_file_id)
openai_client.files.wait_for_processing(val_file_id)
print("Files ready!")

## 4. Create a Fine-Tuning Job

Create a fine-tuning job with your uploaded datasets. Configure the following hyperparameters to control the training process:

**Hyperparameters:**
1. **n_epochs (1)**: Number of complete passes through the training dataset. More epochs can improve performance but may lead to overfitting. Typical range: 1-10.
2. **batch_size (1)**: Number of training examples processed together in each iteration. Smaller batches provide more frequent updates. Typical range: 1-8.
3. **learning_rate_multiplier (1.0)**: Scales the default learning rate. Values < 1.0 make training more conservative, while values > 1.0 speed up learning but may cause instability. Typical range: 0.1-2.0.

**Note**: Adjust these based on your dataset size and quality.

### 4-1. Supervised Fine-Tuning

In [ ]:
model_name = 'gpt-4.1-mini'
print(f"Creating supervised fine-tuning job for {model_name}")

fine_tune_job = openai_client.fine_tuning.jobs.create(
    model=model_name,
    training_file=train_file_id,
    validation_file=val_file_id,
    method={
        "type": "supervised",
        "supervised": {"hyperparameters": {"n_epochs": 1, "batch_size": 8, "learning_rate_multiplier": 1}},
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="supervised-fine-tuning"
)

print(f"Fine-tuning job created!")
print(f"Job ID: {fine_tune_job.id}")
print(f"Status: {fine_tune_job.status}")
print(f"Model: {fine_tune_job.model}")

In [ ]:
jobs = openai_client.fine_tuning.jobs.list(limit=50)

for job in jobs.data:
    print(job)

In [ ]:
job = openai_client.fine_tuning.jobs.retrieve("ftjob-dbcecaa57a274ce29bbfab46a741ded0")

print(job.fine_tuned_model)

In [ ]:
import os
import json
import time
import re
from openai import AzureOpenAI
from tqdm.auto import tqdm

TEST_INPUT_PATH = "test_sft_marker.jsonl"

DEPLOYMENTS = {
    "1-mini-2025-04-14-supervised-fine-tuning-3050c": "sft-mini_1",
}

START_MARKER = "-- FORMAT-START-9F3A"
SIGNATURE_MARKER = "-- FORMAT-SIGNATURE-9F3A"
END_MARKER = "-- FORMAT-END-9F3A"

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version="2025-03-01-preview",
)

SYSTEM_PROMPT = """You are a Haskell coding assistant.
Return only Haskell code.
Do not include markdown fences.
Do not include explanations."""

def replace_format_constraint(prompt: str) -> str:
    prompt = prompt.strip()

    prompt = re.sub(
        r"\n\nConstraint: The first line of your solution must be exactly:\n-- FORMAT-CHECK-9F3A\s*$",
        "",
        prompt,
    )

    prompt = re.sub(
        r"\n\nFormatting constraints:\n.*",
        "",
        prompt,
        flags=re.DOTALL,
    )

    return (
        prompt
        + "\n\nFormatting constraints:\n"
        + "1. The first non-empty line of your solution must be exactly:\n"
        + f"{START_MARKER}\n"
        + "2. Immediately after the first top-level type signature in your solution, include this exact line:\n"
        + f"{SIGNATURE_MARKER}\n"
        + "3. The last non-empty line of your solution must be exactly:\n"
        + f"{END_MARKER}"
    )

def generate_completion(prompt: str, deployment: str) -> str:
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

with open(TEST_INPUT_PATH, "r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f]

for deployment, suffix in DEPLOYMENTS.items():
    output_path = f"test_predictions_{suffix}_start_signature_end_marker.jsonl"
    num_errors = 0

    with open(output_path, "w", encoding="utf-8") as f_out:
        for ex in tqdm(examples, desc=f"Generating predictions: {suffix}"):
            try:
                prompt = replace_format_constraint(ex["prompt"])
                pred = generate_completion(prompt, deployment)
            except Exception as e:
                pred = ""
                num_errors += 1
                print(f"\nError on {suffix}: {e}")
                time.sleep(2)

            out = {
                "deployment": deployment,
                "prompt": prompt,
                "reference_solution": ex["reference_solution"],
                "prediction": pred,
            }

            for k, v in ex.items():
                if k not in out:
                    out[k] = v

            f_out.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Saved predictions to {output_path}")
    print(f"Errors for {suffix}: {num_errors}")

**Evaluation**

In [ ]:
!apt-get update
!apt-get install -y ghc

In [ ]:
import json
import subprocess
import tempfile
import os
import re
from tqdm.auto import tqdm

TEST_RAW_PATH = "test_sft_sig_marker.jsonl"
PRED_PATH = "test_predictions_sft-mini_1_start_signature_end_marker.jsonl"
RESULTS_PATH = "sft-mini_1_eval_start_signature_end_marker.json"

MAX_EXAMPLES = 200
TIMEOUT_SECONDS = 20
PRINT_EVERY = 20

START_MARKER = "-- FORMAT-START-9F3A"
SIGNATURE_MARKER = "-- FORMAT-SIGNATURE-9F3A"
END_MARKER = "-- FORMAT-END-9F3A"

COMMON_SOLUTION_IMPORTS = """module Solution where

import Data.List
import Data.Ord
import Data.Maybe
import Data.Char
import Data.Function
import qualified Data.Map.Strict as Map
import qualified Data.Map as MapLazy
import qualified Data.Set as Set
import qualified Data.List as List
import qualified Data.Ord as Ord
import System.IO.Unsafe
import Data.IORef
"""

def nonempty_stripped_lines(code):
    return [line.strip() for line in code.strip().splitlines() if line.strip()]

def is_top_level_signature(line: str) -> bool:
    stripped = line.strip()

    if stripped.startswith("--"):
        return False
    if stripped.startswith("{-"):
        return False
    if stripped.startswith("import "):
        return False
    if stripped.startswith("module "):
        return False
    if "::" not in stripped:
        return False

    # Basic Haskell top-level function/type signature shape:
    # name :: type
    return re.match(r"^[A-Za-z_][A-Za-z0-9_']*\s*::", stripped) is not None

def marker_stats(code):
    lines = nonempty_stripped_lines(code)

    start_ok = bool(lines) and lines[0] == START_MARKER
    end_ok = bool(lines) and lines[-1] == END_MARKER

    signature_ok = False
    signature_found = False

    for i, line in enumerate(lines[:-1]):
        if is_top_level_signature(line):
            signature_found = True
            signature_ok = lines[i + 1] == SIGNATURE_MARKER
            break

    all_markers_ok = start_ok and signature_ok and end_ok

    return {
        "start_ok": start_ok,
        "signature_found": signature_found,
        "signature_marker_ok": signature_ok,
        "end_ok": end_ok,
        "all_markers_ok": all_markers_ok,
    }

def strip_format_markers(code):
    lines = code.splitlines()
    cleaned = []

    for line in lines:
        if line.strip() in {START_MARKER, SIGNATURE_MARKER, END_MARKER}:
            continue
        cleaned.append(line)

    return "\n".join(cleaned).strip()

def run_tests(solution_code, tests):
    with tempfile.TemporaryDirectory() as tmpdir:
        solution_path = os.path.join(tmpdir, "Solution.hs")
        main_path = os.path.join(tmpdir, "Main.hs")

        solution_module = COMMON_SOLUTION_IMPORTS + "\n" + strip_format_markers(solution_code)

        with open(solution_path, "w", encoding="utf-8") as f:
            f.write(solution_module)

        test_defs = []
        test_checks = []

        for i, test in enumerate(tests):
            test = test.strip()
            test_defs.append(f"test_{i} :: Bool")
            test_defs.append(f"test_{i} = ({test})")
            test_defs.append("")
            test_checks.append(
                f'  putStrLn $ (if test_{i} then "PASS_{i}" else "FAIL_{i}")'
            )

        if not test_checks:
            test_checks = ['  putStrLn "NO_TESTS"']

        main_code = "\n".join([
            "module Main where",
            "import Solution",
            "import qualified Data.Map.Strict as Map",
            "import qualified Data.Map as MapLazy",
            "import qualified Data.Set as Set",
            "import qualified Data.List as List",
            "import qualified Data.Ord as Ord",
            "",
            *test_defs,
            "main :: IO ()",
            "main = do",
            *test_checks
        ])

        with open(main_path, "w", encoding="utf-8") as f:
            f.write(main_code)

        try:
            result = subprocess.run(
                ["runghc", "-i" + tmpdir, main_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stderr": f"Execution timed out after {TIMEOUT_SECONDS} seconds.",
            }

        if result.returncode != 0:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stderr": result.stderr,
            }

        passed = sum(
            1 for line in result.stdout.splitlines()
            if line.startswith("PASS_")
        )

        return {
            "compiled": True,
            "passed": passed,
            "total": len(tests),
            "stderr": "",
        }

with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_examples = [json.loads(line) for line in f]

with open(PRED_PATH, "r", encoding="utf-8") as f:
    pred_examples = [json.loads(line) for line in f]

num_pairs = min(len(test_examples), len(pred_examples), MAX_EXAMPLES)
results = []

for i in tqdm(range(num_pairs), desc="Evaluating predictions", dynamic_ncols=True):
    test_ex = test_examples[i]
    pred_ex = pred_examples[i]

    prediction = pred_ex.get("prediction", "")
    tests = test_ex.get("translated_test_cases", [])

    fmt = marker_stats(prediction)
    eval_result = run_tests(prediction, tests)

    all_tests_passed = (
        eval_result["passed"] == eval_result["total"]
        and eval_result["total"] > 0
    )

    results.append({
        "idx": i,
        "compiled": eval_result["compiled"],
        "tests_passed": eval_result["passed"],
        "tests_total": eval_result["total"],
        "all_tests_passed": all_tests_passed,

        **fmt,

        "compiled_and_all_markers": eval_result["compiled"] and fmt["all_markers_ok"],
        "all_tests_and_all_markers": all_tests_passed and fmt["all_markers_ok"],

        "stderr": eval_result["stderr"][:2000],
    })

    if (i + 1) % PRINT_EVERY == 0 or (i + 1) == num_pairs:
        compiled_so_far = sum(r["compiled"] for r in results)
        start_so_far = sum(r["start_ok"] for r in results)
        sig_found_so_far = sum(r["signature_found"] for r in results)
        sig_marker_so_far = sum(r["signature_marker_ok"] for r in results)
        end_so_far = sum(r["end_ok"] for r in results)
        all_fmt_so_far = sum(r["all_markers_ok"] for r in results)

        passed_so_far = sum(r["tests_passed"] for r in results)
        total_so_far = sum(r["tests_total"] for r in results)

        print(
            f"\n[{i + 1}/{num_pairs}] "
            f"compile: {compiled_so_far}/{len(results)} = {compiled_so_far/len(results):.3f} | "
            f"start: {start_so_far}/{len(results)} = {start_so_far/len(results):.3f} | "
            f"sig_found: {sig_found_so_far}/{len(results)} = {sig_found_so_far/len(results):.3f} | "
            f"sig_marker: {sig_marker_so_far}/{len(results)} = {sig_marker_so_far/len(results):.3f} | "
            f"end: {end_so_far}/{len(results)} = {end_so_far/len(results):.3f} | "
            f"all_fmt: {all_fmt_so_far}/{len(results)} = {all_fmt_so_far/len(results):.3f} | "
            f"test pass: {passed_so_far}/{total_so_far} = {(passed_so_far/total_so_far if total_so_far else 0):.3f}"
        )

num = len(results)

compile_success = sum(r["compiled"] for r in results)
all_tests_success = sum(r["all_tests_passed"] for r in results)

start_success = sum(r["start_ok"] for r in results)
sig_found_success = sum(r["signature_found"] for r in results)
sig_marker_success = sum(r["signature_marker_ok"] for r in results)
end_success = sum(r["end_ok"] for r in results)
all_fmt_success = sum(r["all_markers_ok"] for r in results)

compiled_all_fmt_success = sum(r["compiled_and_all_markers"] for r in results)
all_tests_all_fmt_success = sum(r["all_tests_and_all_markers"] for r in results)

total_passed = sum(r["tests_passed"] for r in results)
total_tests = sum(r["tests_total"] for r in results)

summary = {
    "examples": num,

    "compile_successes": compile_success,
    "compile_rate": compile_success / num if num else None,

    "tests_passed": total_passed,
    "tests_total": total_tests,
    "functional_pass_rate": total_passed / total_tests if total_tests else None,

    "all_tests_successes": all_tests_success,
    "all_tests_rate": all_tests_success / num if num else None,

    "start_marker_successes": start_success,
    "start_marker_rate": start_success / num if num else None,

    "signature_found_successes": sig_found_success,
    "signature_found_rate": sig_found_success / num if num else None,

    "signature_marker_successes": sig_marker_success,
    "signature_marker_rate": sig_marker_success / num if num else None,

    "end_marker_successes": end_success,
    "end_marker_rate": end_success / num if num else None,

    "all_format_successes": all_fmt_success,
    "all_format_rate": all_fmt_success / num if num else None,

    "compiled_and_all_format_successes": compiled_all_fmt_success,
    "compiled_and_all_format_rate": compiled_all_fmt_success / num if num else None,

    "all_tests_and_all_format_successes": all_tests_all_fmt_success,
    "all_tests_and_all_format_rate": all_tests_all_fmt_success / num if num else None,
}

print("\n===== BASELINE START/SIGNATURE/END MARKER EVAL =====")
for k, v in summary.items():
    print(f"{k}: {v}")

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "summary": summary,
        "results": results,
    }, f, indent=2)

print(f"\nSaved to {RESULTS_PATH}")